# Python Tables Course

A self-paced, approximately two-hour course on tabular data with **pandas**. Each row represents one team; each column stores one variable. You will build league standings from the supplied 2015–16 season records, combine them with team information, and interpret simple exploratory plots.

**Files:** `EPLresults.csv`, `EPLteams.csv`, and `EPLFinalStandings.csv` must be beside this notebook. Run cells from top to bottom. The supplied final standings are a check, not an input to the calculation.

**Prerequisites:** basic Python variables, lists, and arithmetic. You need `pandas`, `matplotlib`, and Jupyter. In a terminal, install missing packages with `python -m pip install pandas matplotlib jupyter`.

**Learning outcomes:** import and inspect CSV data; select, filter, and sort rows; calculate columns; join datasets; clean names, dates, and categories; export a CSV; and distinguish an observed association from a causal claim.

In [2]:
# Load the libraries and resolve data paths relative to the notebook folder.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd()
results_path = DATA_DIR / "EPLresults.csv"
teams_path = DATA_DIR / "EPLteams.csv"
reference_path = DATA_DIR / "EPLFinalStandings.csv"
for path in (results_path, teams_path, reference_path):
    if not path.is_file():
        raise FileNotFoundError(f"Place {path.name} beside this notebook and run it there.")

## 1. Load and inspect a table

`pd.read_csv` reads headers into column names and infers types. Inspect shape, names, missing values, and numeric summaries before changing the data. Team names are text; match statistics are integers.

In [3]:
# Read the season results and inspect their structure.
results = pd.read_csv(results_path)
print(results.shape)
print(results.head(3).to_string(index=False))
print(results.dtypes.to_string())
print(results.isna().sum().to_string())

(20, 11)
       Team  Home Wins  Home Draws  Home Losses  Home GF  Home GA  Away Wins  Away Draws  Away Losses  Away GF  Away GA
    Arsenal         12           4            3       31       11          8           7            4       34       25
Aston Villa          2           5           12       14       35          1           3           15       13       41
Bournemouth          5           5            9       23       34          6           4            9       22       33
Team           object
Home Wins       int64
Home Draws      int64
Home Losses     int64
Home GF         int64
Home GA         int64
Away Wins       int64
Away Draws      int64
Away Losses     int64
Away GF         int64
Away GA         int64
Team           0
Home Wins      0
Home Draws     0
Home Losses    0
Home GF        0
Home GA        0
Away Wins      0
Away Draws     0
Away Losses    0
Away GF        0
Away GA        0


In [4]:
# Summarize the numeric columns and identify the home and away fields.
print(results.describe().to_string())
home_cols = [name for name in results.columns if name.startswith("Home ")]
away_cols = [name for name in results.columns if name.startswith("Away ")]
print("Home:", home_cols)
print("Away:", away_cols)

       Home Wins  Home Draws  Home Losses    Home GF    Home GA  Away Wins  Away Draws  Away Losses    Away GF    Away GA
count  20.000000   20.000000    20.000000  20.000000  20.000000  20.000000   20.000000    20.000000  20.000000  20.000000
mean    7.850000    5.350000     5.800000  28.350000  22.950000   5.800000    5.350000     7.850000  22.950000  28.350000
std     2.870448    1.694418     2.876401   8.235194   6.886103   2.462348    1.871532     3.587625   6.977219   7.513322
min     2.000000    2.000000     1.000000  14.000000   9.000000   1.000000    2.000000     2.000000  12.000000  18.000000
25%     6.000000    4.750000     3.000000  21.500000  19.750000   4.000000    4.000000     5.000000  19.750000  22.750000
50%     7.500000    5.000000     5.000000  29.000000  22.500000   6.000000    5.500000     7.500000  22.000000  27.000000
75%    10.250000    6.000000     8.000000  34.250000  27.000000   7.000000    7.000000    10.000000  27.750000  32.250000
max    12.000000    9.00

**Try it:** How many teams and columns are present? Are any values missing? Find the largest number of home wins. Check with the next cell.

In [ ]:
# Check the first inspection questions against the supplied file.
assert results.shape == (20, 11)
assert not results.isna().any().any()
print("Most home wins:", results["Home Wins"].max())

## 2. Sort, select, and filter

`sort_values` accepts one or several column names. `ascending=False` puts large values first. `loc` selects by row labels and column names; `iloc` selects by integer positions. A one-column selection returns a `Series`; a list of columns returns a `DataFrame`. Row positions start at zero.

In [ ]:
# Sort by two performance measures and preserve all columns together.
by_wins = results.sort_values(["Home Wins", "Away Wins"], ascending=[False, False])
print(by_wins[["Team", "Home Wins", "Away Wins"]].head(6).to_string(index=False))

In [ ]:
# Select columns by name, rows by position, and teams by a Boolean condition.
team_wins = results.loc[:, ["Team", "Home Wins", "Away Wins"]]
first_three = team_wins.iloc[:3]
manchester = team_wins.loc[team_wins["Team"].str.startswith("Manchester")]
print(first_three.to_string(index=False))
print(manchester.to_string(index=False))

**Try it:** Show the team name and away goals for the clubs with at least 10 away wins, ordered by away goals. Decide whether the comparison is fair without considering matches played.

In [ ]:
# Check the away-win filter and keep only the requested fields.
strong_away = results.loc[results["Away Wins"] >= 10, ["Team", "Away Wins", "Away GF"]]
print(strong_away.sort_values("Away GF", ascending=False).to_string(index=False))

## 3. Calculate standings

For each statistic, add the home and away columns **row by row**. Assigning a `Series` adds a new column. League points are three per win plus one per draw. Goal difference is goals scored minus goals conceded. Sort by points, then goal difference. A complete competition rule may require further tie breakers; this dataset needs only these two to reproduce the supplied order.

In [ ]:
# Build a new table without changing the imported results.
standings = pd.DataFrame({"Team": results["Team"]})
for statistic in ("Wins", "Draws", "Losses", "GF", "GA"):
    standings[f"Total{statistic}"] = results[f"Home {statistic}"] + results[f"Away {statistic}"]
standings["GoalDiff"] = standings["TotalGF"] - standings["TotalGA"]
standings["Points"] = 3 * standings["TotalWins"] + standings["TotalDraws"]
standings = standings.sort_values(["Points", "GoalDiff"], ascending=[False, False]).reset_index(drop=True)
print(standings.head(6).to_string(index=False))

In [ ]:
# Validate the arithmetic and compare the result with the supplied standings.
assert (standings["TotalWins"] + standings["TotalDraws"] + standings["TotalLosses"] == 38).all()
assert standings.iloc[0]["Team"] == "Leicester City"
reference = pd.read_csv(reference_path)
pd.testing.assert_frame_equal(standings, reference, check_dtype=False)
print("All 20 rows and every standings column match the supplied reference.")

**Try it:** Find the teams with 66 points and explain their ordering. Then list teams with a negative goal difference but at least 45 points.

In [ ]:
# Examine a tie and filter standings using two conditions.
print(standings.loc[standings["Points"] == 66, ["Team", "Points", "GoalDiff"]].to_string(index=False))
print(standings.loc[(standings["GoalDiff"] < 0) & (standings["Points"] >= 45), ["Team", "Points", "GoalDiff"]].to_string(index=False))

## 4. Join information from another table

The team file contains payroll, manager, hire date, and nationality. `merge` matches rows using the `Team` key. `validate="one_to_one"` catches duplicate keys; `indicator=True` exposes unmatched rows. Check the result before dropping the indicator.

In [ ]:
# Read team information and verify that each team appears only once in each file.
teams = pd.read_csv(teams_path)
assert standings["Team"].is_unique and teams["Team"].is_unique
joined = standings.merge(teams, on="Team", how="left", validate="one_to_one", indicator=True)
print(joined["_merge"].value_counts().to_string())
assert joined["_merge"].eq("both").all()
joined = joined.drop(columns="_merge")
print(joined.head(3).to_string(index=False))

**Try it:** If a team were absent from `EPLteams.csv`, what would a left join place in its team-information cells? Which check above would detect that case?

## 5. Clean column names and data types

Readable labels can contain spaces and symbols. Rename a column when a shorter name helps downstream code. Parse dates with an explicit format and convert a repeated text field to a category. Keep the original hire dates available until you have checked the conversion.

In [ ]:
# Rename fields and parse the day-month-year text explicitly.
analysis = joined.rename(columns={
    "Payroll (M£)": "Payroll_MGBP",
    "Manager Hire Date": "ManagerHireDate",
    "Manager Nationality": "ManagerNationality",
})
analysis["ManagerHireDate"] = pd.to_datetime(analysis["ManagerHireDate"], format="%d %b %Y", errors="raise")
analysis["ManagerNationality"] = analysis["ManagerNationality"].astype("category")
print(analysis[["Team", "Payroll_MGBP", "ManagerHireDate", "ManagerNationality"]].head().to_string(index=False))
print(analysis[["Payroll_MGBP", "ManagerHireDate", "ManagerNationality"]].dtypes.to_string())

In [ ]:
# Reorder useful fields and demonstrate safe removal on a copy.
front = ["Team", "Points", "GoalDiff", "Payroll_MGBP"]
compact = analysis.loc[:, front + [name for name in analysis.columns if name not in front]].copy()
without_manager = compact.drop(columns=["Manager"])
print(compact.columns.tolist())
print("Manager removed from copy:", "Manager" not in without_manager.columns)

## 6. Export and inspect a CSV

`to_csv(index=False)` avoids writing the DataFrame's row index as an extra data column. Reopen the file to check that the exported schema survives. The output is written beside this notebook.

In [ ]:
# Export the calculated standings and verify the round trip.
output_path = DATA_DIR / "PythonFinalStandings.csv"
standings.to_csv(output_path, index=False)
reloaded = pd.read_csv(output_path)
pd.testing.assert_frame_equal(reloaded, standings, check_dtype=False)
print("Saved", output_path.name, "with", len(reloaded), "teams.")

## 7. Explore relationships carefully

These plots describe the 20 teams in one season. They do not establish that payroll or manager nationality caused a points total. Several managers in the file were hired after the season began, so a tenure calculation at the season start would be negative and cannot describe their full-season tenure. Check dates before using them as explanatory variables.

In [ ]:
# Plot payroll against league points with labels for selected teams.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(analysis["Payroll_MGBP"], analysis["Points"])
for name in ("Leicester City", "Manchester City", "Arsenal"):
    row = analysis.loc[analysis["Team"] == name].iloc[0]
    ax.annotate(name, (row["Payroll_MGBP"], row["Points"]), xytext=(4, 4), textcoords="offset points")
ax.set(xlabel="Payroll (million GBP)", ylabel="League points", title="Payroll and points, 2015–16")
fig.tight_layout()
plt.show()

In [ ]:
# Check which recorded hire dates are later than the season start.
season_start = pd.Timestamp("2015-08-08")
later_hires = analysis.loc[analysis["ManagerHireDate"] > season_start, ["Team", "Manager", "ManagerHireDate"]]
print(later_hires.to_string(index=False))
print("Payroll–points Pearson correlation:", round(analysis["Payroll_MGBP"].corr(analysis["Points"]), 3))

**Try it:** Plot `TotalGA` against `Points`. Describe the pattern without claiming that one variable caused the other. Which team is an interesting exception?

In [ ]:
# Compare goals conceded with points in the supplied season.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(analysis["TotalGA"], analysis["Points"])
ax.set(xlabel="Goals conceded", ylabel="League points", title="Defensive record and points")
fig.tight_layout()
plt.show()

## 8. Final project

Starting from `EPLresults.csv` and `EPLteams.csv`, create a report with one row per team and the columns `Team`, `Points`, `GoalDiff`, `Payroll_MGBP`, and `ManagerNationality`. Sort it by points and goal difference. Save it as `PythonTeamReport.csv`. Before exporting, verify 20 unique teams, no missing join values, and the champion's 81 points. Write three sentences: one factual observation, one limitation, and one question for further study.

The following cell is a complete checkable solution. Attempt the project first, then run it.

In [ ]:
# Build and validate the project report from the earlier calculated tables.
project = standings.merge(teams, on="Team", how="left", validate="one_to_one")
project = project.rename(columns={"Payroll (M£)": "Payroll_MGBP", "Manager Nationality": "ManagerNationality"})
report = project[["Team", "Points", "GoalDiff", "Payroll_MGBP", "ManagerNationality"]].copy()
report = report.sort_values(["Points", "GoalDiff"], ascending=[False, False]).reset_index(drop=True)
assert len(report) == 20 and report["Team"].is_unique
assert not report.isna().any().any()
assert report.iloc[0]["Team"] == "Leicester City" and report.iloc[0]["Points"] == 81
report.to_csv(DATA_DIR / "PythonTeamReport.csv", index=False)
print(report.head(5).to_string(index=False))

## Quick reference

| Goal | pandas expression |
|---|---|
| Import CSV | `pd.read_csv("file.csv")` |
| Inspect | `df.head()`, `df.info()`, `df.describe()` |
| Select columns | `df.loc[:, ["Team", "Points"]]` |
| Select first rows | `df.iloc[:5]` |
| Filter | `df.loc[df["Points"] >= 60]` |
| Sort | `df.sort_values(["Points", "GoalDiff"], ascending=[False, False])` |
| Add a column | `df["GoalDiff"] = df["TotalGF"] - df["TotalGA"]` |
| Join | `left.merge(right, on="Team", validate="one_to_one")` |
| Rename or remove | `df.rename(columns={"old": "new"})`, `df.drop(columns=["old"])` |
| Parse dates | `pd.to_datetime(df["date"], format="%d %b %Y")` |
| Export CSV | `df.to_csv("output.csv", index=False)` |

**Completion check:** You can reproduce the standings, explain the 66-point tie, account for every join row, and distinguish a descriptive plot from a causal claim.